# 🤖 Phase 4: ML Modeling — XGBoost & LightGBM with SHAP
## Credit Risk Analytics — Home Credit Default Risk
---
**Goal:** Build credit default prediction models using XGBoost and LightGBM, handle class imbalance with SMOTE, evaluate with AUC-ROC, and explain predictions using SHAP values.


In [ ]:
# Install required libraries
# !pip install xgboost lightgbm shap imbalanced-learn scikit-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, roc_auc_score,
                              roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import lightgbm as lgb
import shap

print("All libraries loaded ✅")


## 1. Load & Prepare Data

In [ ]:
df = pd.read_csv('application_clean.csv')
print(f"Shape: {df.shape}")
print(f"Default rate: {df['TARGET'].mean()*100:.2f}%")


In [ ]:
# Select features for modeling
drop_cols = ['SK_ID_CURR', 'TARGET', 'AGE_GROUP', 'INCOME_BRACKET']
feature_cols = [c for c in df.select_dtypes(include='number').columns
                if c not in drop_cols]

X = df[feature_cols].copy()
y = df['TARGET'].copy()

print(f"Features: {len(feature_cols)}")
print(f"Class distribution:\n{y.value_counts()}")


## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")
print(f"Train default rate: {y_train.mean()*100:.2f}%")
print(f"Test default rate: {y_test.mean()*100:.2f}%")


## 3. Handle Class Imbalance — SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_train_sm).value_counts().to_dict()}")


## 4. XGBoost Model

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=50
)

# Evaluate
xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_proba)

print(f"\nXGBoost AUC-ROC: {xgb_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, xgb_preds))


## 5. LightGBM Model

In [ ]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Evaluate
lgb_preds = lgb_model.predict(X_test)
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc = roc_auc_score(y_test, lgb_proba)

print(f"LightGBM AUC-ROC: {lgb_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lgb_preds))


## 6. ROC Curve Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curves
for proba, label, color in [
    (xgb_proba, f'XGBoost (AUC={xgb_auc:.3f})', 'steelblue'),
    (lgb_proba, f'LightGBM (AUC={lgb_auc:.3f})', 'crimson')
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=label, color=color, linewidth=2)

axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve Comparison')
axes[0].legend()
axes[0].grid(True)

# Confusion Matrix — LightGBM (best model)
cm = confusion_matrix(y_test, lgb_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Repaid','Defaulted'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('LightGBM Confusion Matrix')

plt.tight_layout()
plt.savefig('plot_roc_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Feature Importance

In [ ]:
# LightGBM feature importance
fi = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barh(fi['feature'], fi['importance'], color='steelblue', edgecolor='black')
plt.title('LightGBM — Top 20 Feature Importances')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 features:")
print(fi.head(10).to_string(index=False))


## 8. SHAP Explainability

In [ ]:
# SHAP values for LightGBM
print("Computing SHAP values (this may take a minute)...")
explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_test[:500])

# Handle binary classification output
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# SHAP Summary Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_test[:500], plot_type='bar', show=False)
plt.title('SHAP Feature Importance (Mean |SHAP value|)')
plt.tight_layout()
plt.savefig('plot_shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# SHAP Beeswarm (detailed impact)
shap.summary_plot(shap_vals, X_test[:500], show=False)
plt.title('SHAP Beeswarm Plot — Feature Impact on Default Prediction')
plt.tight_layout()
plt.savefig('plot_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Model Summary

In [ ]:
print("=" * 50)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 50)
print(f"XGBoost  AUC-ROC: {xgb_auc:.4f}")
print(f"LightGBM AUC-ROC: {lgb_auc:.4f}")
print(f"\nBest model: {'LightGBM' if lgb_auc > xgb_auc else 'XGBoost'}")
print(f"\nTop 5 default predictors (SHAP):")
print(fi.head(5)['feature'].tolist())
print("=" * 50)


In [ ]:
# Save best model
import joblib
joblib.dump(lgb_model, 'lgb_credit_risk_model.pkl')
print("Model saved: lgb_credit_risk_model.pkl ✅")
